# EDA and Data Preparation

## Analysis Goal

- country
- year
- house_price_index
- rent_price_index
- house_price_to_income_ratio
- housing_cost_overburden_rate
- fertility_rate
- life_satisfaction
- living_with_parents
- net_migration_rate
- population

In [258]:
import pandas as pd
from functools import reduce


## Data Understanding


### Data Collection


#### Load data

In [259]:
def read_tsv(file, keep=None):
    df = pd.read_csv(f'eurostat_data/{file}', sep='\t', na_values=[':', ': ', ': @N'])
    first_column = df.columns[0]
    cols = first_column.replace('\\TIME_PERIOD', '').split(',')

    df[cols] = df[first_column].str.split(',', expand=True)
    df = df.drop(columns=first_column)
    
    df = df.rename(columns={'geo': 'country'})
    keep = keep or []

    cols = [col for col in df.columns if (col not in cols and col != 'country')]
    df[cols] = df[cols].replace(r'\s*[A-Za-z]+$', '', regex=True)
    df[cols] = df[cols].replace(['', ':', ': ', ': @N'], pd.NA)
    df[cols] = df[cols].apply(pd.to_numeric)

    ordered_cols = ['country'] + keep + cols
    return df[ordered_cols]

In [260]:
house_price_df = read_tsv('house.tsv')
rent_price_df = read_tsv('rent.tsv')
price_to_income_df = read_tsv('income.tsv')
housing_burden_df = read_tsv('burden.tsv')
population_df =  read_tsv('population.tsv')
migration_df = read_tsv('migration.tsv')
lifesatisfaction_df = read_tsv('lifesatisfaction.tsv')
withparents_df = read_tsv('withparents.tsv')
fertility_df = read_tsv('child.tsv', keep=['indic_de'])

In [261]:
house_price_df.head()

,country,2005,2006,2007,2008,2009,2010,2011,2012,2013,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AT,NaN,NaN,NaN,NaN,NaN,77.32,81.47,86.86,91.20,...,106.71,112.14,118.83,125.97,135.52,151.02,168.49,163.68,163.06,167.33
1,BE,72.85,80.00,86.20,89.95,89.58,91.60,94.88,97.11,98.55,...,102.33,105.95,109.15,113.23,118.09,125.96,132.93,136.05,140.42,144.86
2,BG,80.09,91.86,118.42,147.97,117.79,105.81,99.98,98.07,95.92,...,107.02,116.30,123.96,131.42,137.41,149.33,169.92,186.67,217.47,249.23
3,CY,97.06,108.57,121.32,128.17,119.76,112.90,111.09,107.70,103.29,...,99.82,104.48,103.08,111.70,117.49,115.70,123.22,134.60,144.46,150.89
4,CZ,NaN,NaN,NaN,100.80,96.90,95.20,95.30,93.90,93.90,...,107.20,119.70,130.00,141.90,153.90,184.20,215.30,211.70,222.30,245.40


In [262]:
rent_price_df.head()

,country,1996-01,1996-02,1996-03,1996-04,1996-05,1996-06,1996-07,1996-08,1996-09,...,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
0,AL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,117.03,118.15,118.68,120.01,120.78,121.45,123.14,123.18,123.20,125.34
1,AT,54.21,54.32,54.43,54.60,54.77,54.99,55.43,55.93,56.43,...,145.20,145.57,145.83,146.43,147.52,148.16,148.96,149.97,149.87,149.84
2,BE,72.42,72.42,72.35,72.42,72.50,72.50,72.50,72.50,72.57,...,128.77,128.85,129.64,130.34,130.24,130.65,130.50,132.39,132.37,132.91
3,BG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,144.11,145.75,145.74,146.87,149.45,152.07,152.30,153.60,153.76,153.92
4,CH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,112.90,113.03,113.36,113.49,113.62,114.04,114.16,114.27,114.17,114.31


In [263]:
price_to_income_df.head()

,country,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,AT,86.02,85.83,85.08,82.51,78.99,85.87,85.37,84.89,83.25,...,100.0,103.89,106.92,109.83,114.07,121.91,128.75,132.93,121.77,113.29
1,BE,71.54,71.85,75.11,78.82,83.63,91.59,96.17,99.29,98.47,...,100.0,99.89,100.21,100.24,100.39,103.29,104.36,101.34,96.43,96.95
2,BG,131.99,118.71,93.96,98.39,132.95,163.46,164.72,189.34,192.83,...,100.0,101.50,101.65,96.34,95.78,98.18,95.34,91.00,86.60,78.64
3,CY,96.87,93.09,96.77,99.86,100.20,100.75,106.15,113.08,108.47,...,100.0,95.76,95.42,89.35,88.50,91.53,85.81,86.93,89.67,90.20
4,CZ,90.68,92.34,98.03,102.98,98.01,96.00,96.49,109.00,114.00,...,100.0,103.13,106.93,107.94,110.19,112.36,122.58,131.30,119.31,119.40


In [264]:
housing_burden_df.head()

,country,2003,2004,2005,2006,2007,2008,2009,2010,2011,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.2,6.0,5.5,3.8,3.3,2.8,2.1,NaN,NaN
1,AT,NaN,7.9,4.6,5.0,5.4,8.5,8.0,7.5,7.8,...,7.2,7.1,6.8,7.0,6.3,6.1,7.4,6.0,6.3,6.5
2,BE,NaN,12.6,8.6,9.8,10.1,12.5,8.7,8.9,10.6,...,9.8,9.4,8.9,8.4,7.7,7.6,7.5,7.8,6.7,6.7
3,BG,NaN,NaN,NaN,15.4,21.2,13.3,7.0,5.9,8.7,...,20.7,18.9,17.9,16.0,14.4,11.6,15.1,11.1,8.0,6.9
4,CH,NaN,NaN,NaN,NaN,NaN,NaN,13.2,14.1,13.1,...,12.0,12.7,12.8,14.2,13.6,14.8,13.4,14.2,15.3,NaN


In [265]:
population_df.head()

,country,1960,1961,1962,1963,1964,1965,1966,1967,1968,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,76177.0,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,1583800.0,1633800.0,1685800.0,1736838.0,1788404.0,1839866.0,1889715.0,1939430.0,1991765.0,...,2875592.0,2876591.0,2870324.0,2862427.0,2845955.0,2829741.0,2793592.0,2761785.0,NaN,2363314.0
2,AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2998577.0,2986151.0,2972732.0,2965269.0,2959694.0,2963251.0,NaN,2977130.0,2991201.0,NaN
3,AT,7030385.0,7064693.0,7107904.0,7151824.0,7199798.0,7247804.0,7293973.0,7350159.0,7403837.0,...,8700471.0,8772865.0,8822267.0,8858775.0,8901064.0,8932664.0,8978929.0,9104772.0,9158750.0,9197213.0
4,AZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9705643.0,9809981.0,9898085.0,9981457.0,10067108.0,10119133.0,10156366.0,10127145.0,10180770.0,10224889.0


In [266]:
migration_df.head()

,country,1960,1961,1962,1963,1964,1965,1966,1967,1968,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1049.0,NaN,1175.0,1128.0,NaN,NaN,1922.0,3412.0,NaN,NaN
1,AL,-2911.0,-1007.0,2192.0,245.0,-1326.0,888.0,1057.0,-361.0,5358.0,...,-9346.0,-14904.0,-15027.0,-23096.0,-16684.0,-32853.0,-32497.0,-373174.0,-28836.0,NaN
2,AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-24792.0,-23962.0,-18286.0,-15430.0,3374.0,NaN,NaN,1794.0,76893.0,NaN
3,AT,-2034.0,-2679.0,1521.0,4744.0,3246.0,10518.0,19049.0,21712.0,-6970.0,...,65388.0,45039.0,34948.0,40723.0,39596.0,52149.0,136548.0,66133.0,49711.0,NaN
4,AZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1522.0,1172.0,1640.0,388.0,1101.0,1827.0,-91257.0,1155.0,718.0,NaN


In [267]:
lifesatisfaction_df.head()

,country,2013,2018,2021,2022,2023,2024,2025
0,AL,NaN,5.5,5.7,6.0,6.3,NaN,NaN
1,AT,7.8,8.0,8.0,7.9,7.7,7.6,7.7
2,BE,7.5,7.6,7.5,7.6,7.7,7.6,7.6
3,BG,4.8,5.4,5.7,5.6,5.9,6.2,6.3
4,CH,8.0,8.0,7.9,8.0,7.8,7.7,NaN


In [268]:
withparents_df.head()

,country,2003,2004,2005,2006,2007,2008,2009,2010,2011,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,72.7,74.5,71.3,75.3,76.8,76.1,76.4,NaN,NaN
1,AT,42.3,40.3,44.5,45.8,47.0,46.0,47.4,46.9,46.5,...,42.6,42.1,40.9,42.6,40.4,37.7,38.6,40.0,39.0,40.3
2,BE,38.1,38.5,39.3,42.5,41.5,42.2,41.4,39.5,40.9,...,47.3,47.2,48.5,47.5,48.2,47.1,45.3,46.3,48.3,51.4
3,BG,NaN,NaN,58.1,58.1,59.7,62.7,62.8,66.0,66.7,...,58.9,58.4,59.1,59.2,59.3,60.1,59.6,59.7,58.7,57.8
4,CH,NaN,NaN,NaN,NaN,36.9,41.1,41.2,43.5,42.9,...,39.3,37.9,42.1,38.9,38.6,38.5,38.0,40.6,42.4,NaN


In [269]:
fertility_df.head()

,country,indic_de,1960,1961,1962,1963,1964,1965,1966,1967,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,AD,AGEMOTH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,AGEMOTH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,27.9,28.2,28.4,28.6,NaN,29.0,29.2,29.5,NaN,29.6
2,AM,AGEMOTH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.8,27.0,27.0,27.1,27.3,NaN,NaN,NaN,NaN,NaN
3,AT,AGEMOTH,27.6,27.5,27.5,27.4,27.4,27.3,27.1,27.0,...,30.6,30.6,30.7,30.9,31.0,31.0,31.2,31.2,31.2,31.2
4,AZ,AGEMOTH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,25.6,25.5,25.6,25.6,25.6,NaN,NaN,NaN,NaN,NaN


In [270]:
fertility_df.tail()

,country,indic_de,1960,1961,1962,1963,1964,1965,1966,1967,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
93,SM,TOTFERRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,TR,TOTFERRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.14,2.11,2.07,1.99,1.88,NaN,NaN,1.63,1.51,1.48
95,UA,TOTFERRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.36,1.35,1.26,1.20,1.14,NaN,NaN,NaN,NaN,NaN
96,UK,TOTFERRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.80,1.79,1.74,1.68,NaN,NaN,NaN,NaN,NaN,NaN
97,XK,TOTFERRT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.66,1.65,1.61,1.55,NaN,NaN,NaN,NaN,NaN


####  Transform dataframe format

In [271]:
def transform_to_long(df, name, indicator_col=None):
    value_per_year = []
    for col in df.columns:
        column_string = str(col).strip()
        if column_string.isdigit():
            value_per_year.append(col)


    id_cols = ['country']
    if indicator_col is not None:
        id_cols.append(indicator_col)

    long_df = df.melt(
        id_vars=id_cols,
        value_vars=value_per_year,
        var_name='year',
        value_name=name
    )

    if indicator_col is not None:
        long_df = long_df.pivot_table(
            index=['country', 'year'],
            columns=indicator_col,
            values=name,
            aggfunc='first'
        ).reset_index()

        long_df = long_df.rename(columns = { 'AGEMOTH': 'mean_childbirth_age',  
                                            'TOTFERRT': 'fertility_rate'})
        

    return long_df


In [272]:
def transform_to_long_and_yearly(df):

    value_per_month = []
    for col in df.columns:
        if '-' in str(col):
            value_per_month.append(col)

    rent_long = df.melt(
        id_vars=['country'],
        value_vars=value_per_month,
        var_name='month',
        value_name='rent_price_index'
    )

    rent_long['year'] = rent_long['month'].str[:4]

    rent_yearly = (
        rent_long
        .groupby(['country', 'year'], as_index=False)['rent_price_index']
        .mean()
    )

    return rent_yearly

In [273]:
income_long = transform_to_long(price_to_income_df, 'price_to_income_index')
burden_long = transform_to_long(housing_burden_df, 'housing_burden')
house_price_long =transform_to_long(house_price_df, 'house_price_index')
population_long = transform_to_long(population_df, 'population')
migration_long = transform_to_long(migration_df, 'net_migration')
lifesatisfaction_long = transform_to_long(lifesatisfaction_df, 'life_satisfaction')
withparents_long = transform_to_long(withparents_df, 'living_with_parents')
fertility_long = transform_to_long(fertility_df, 'value', indicator_col='indic_de')
rent_long = transform_to_long_and_yearly(rent_price_df)


#### Standardize the date range

In [274]:
print(f'House price to income ratio: {min(income_long['year'])}- {max(income_long['year'])}')
print(f'Housing burden: {min(burden_long['year'])}- {max(burden_long['year'])}')
print(f'House price index: {min(house_price_long['year'])}- {max(house_price_long['year'])}')
print(f'Rent price index: {min(rent_long['year'])}- {max(rent_long['year'])}')

print(f'Population: {min(population_long['year'])}- {max(population_long['year'])}')
print(f'Net migration: {min(migration_long['year'])}- {max(migration_long['year'])}')
print(f'Living with parents: {min(withparents_long['year'])}- {max(withparents_long['year'])}')
print(f'Fertility: {min(fertility_long['year'])}- {max(fertility_long['year'])}')
print(f'Life satisfaction: {min(lifesatisfaction_long['year'])}- {max(lifesatisfaction_long['year'])}')


House price to income ratio: 2000 - 2024 
Housing burden: 2003 - 2025 
House price index: 2005 - 2025 
Rent price index: 1996- 2025
Population: 1960 - 2025 
Net migration: 1960 - 2025 
Living with parents: 2003 - 2025 
Fertility: 1960 - 2024 
Life satisfaction: 2013 - 2025 


In [275]:
START_YEAR = 2005
END_YEAR = 2024

def filter_years(df):
    df['year'] = df['year'].astype(int)
    return df[(df['year'] >= START_YEAR) & (df['year'] <= END_YEAR)]

income_long = filter_years(income_long)
burden_long = filter_years(burden_long)
house_price_long = filter_years(house_price_long)
rent_long = filter_years(rent_long)
population_long = filter_years(population_long)
migration_long = filter_years(migration_long)
withparents_long = filter_years(withparents_long)
fertility_long = filter_years(fertility_long)
lifesatisfaction_long = filter_years(lifesatisfaction_long)

#### Merge

In [276]:
final_df = income_long.merge(burden_long, on=["country", "year"], how="outer")
final_df = final_df.merge(house_price_long, on=["country", "year"], how="outer")
final_df = final_df.merge(population_long, on=["country", "year"], how="outer")
final_df = final_df.merge(migration_long, on=["country", "year"], how="outer")
final_df = final_df.merge(lifesatisfaction_long, on=["country", "year"], how="outer")
final_df = final_df.merge(withparents_long, on=["country", "year"], how="outer")
final_df = final_df.merge(fertility_long, on=["country", "year"], how="outer")
final_df = final_df.merge(rent_long, on=["country", "year"], how="outer")

In [277]:
final_df.head()

,country,year,price_to_income_index,housing_burden,house_price_index,population,net_migration,life_satisfaction,living_with_parents,mean_childbirth_age,fertility_rate,rent_price_index
0,AD,2005,NaN,NaN,NaN,76875.0,1122.0,NaN,NaN,NaN,NaN,NaN
1,AD,2006,NaN,NaN,NaN,78549.0,2090.0,NaN,NaN,30.3,1.24,NaN
2,AD,2007,NaN,NaN,NaN,81222.0,1319.0,NaN,NaN,30.4,1.18,NaN
3,AD,2008,NaN,NaN,NaN,83137.0,709.0,NaN,NaN,30.5,1.25,NaN
4,AD,2009,NaN,NaN,NaN,84484.0,-968.0,NaN,NaN,31.1,1.19,NaN


In [278]:
len(final_df)

1100

###  Data Description and Initial Exploration

In [279]:
final_df.shape

(1100, 12)

In [280]:
final_df.head()

,country,year,price_to_income_index,housing_burden,house_price_index,population,net_migration,life_satisfaction,living_with_parents,mean_childbirth_age,fertility_rate,rent_price_index
0,AD,2005,NaN,NaN,NaN,76875.0,1122.0,NaN,NaN,NaN,NaN,NaN
1,AD,2006,NaN,NaN,NaN,78549.0,2090.0,NaN,NaN,30.3,1.24,NaN
2,AD,2007,NaN,NaN,NaN,81222.0,1319.0,NaN,NaN,30.4,1.18,NaN
3,AD,2008,NaN,NaN,NaN,83137.0,709.0,NaN,NaN,30.5,1.25,NaN
4,AD,2009,NaN,NaN,NaN,84484.0,-968.0,NaN,NaN,31.1,1.19,NaN


In [281]:
final_df.columns

Index(['country', 'year', 'price_to_income_index', 'housing_burden',
       'house_price_index', 'population', 'net_migration', 'life_satisfaction',
       'living_with_parents', 'mean_childbirth_age', 'fertility_rate',
       'rent_price_index'],
      dtype='object')

In [282]:
final_df['year'].min(), final_df['year'].max()

(np.int64(2005), np.int64(2024))

In [283]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   country                1100 non-null   object 
 1   year                   1100 non-null   int64  
 2   price_to_income_index  540 non-null    float64
 3   housing_burden         656 non-null    float64
 4   house_price_index      552 non-null    float64
 5   population             974 non-null    float64
 6   net_migration          980 non-null    float64
 7   life_satisfaction      203 non-null    float64
 8   living_with_parents    666 non-null    float64
 9   mean_childbirth_age    847 non-null    float64
 10  fertility_rate         847 non-null    float64
 11  rent_price_index       727 non-null    float64
dtypes: float64(10), int64(1), object(1)
memory usage: 103.3+ KB


In [284]:
final_df.describe()

,year,price_to_income_index,housing_burden,house_price_index,population,net_migration,life_satisfaction,living_with_parents,mean_childbirth_age,fertility_rate,rent_price_index
count,1100.000000,540.000000,656.000000,552.000000,9.740000e+02,9.800000e+02,203.000000,666.000000,847.000000,847.000000,727.000000
mean,2014.500000,108.566167,9.677896,122.833551,3.317420e+07,7.062668e+04,7.067980,49.924625,29.806612,1.561558,102.023918
std,5.768904,19.731653,6.192129,92.317101,8.870763e+07,2.311856e+05,0.700996,16.025744,1.599358,0.223232,36.694299
min,2005.000000,55.880000,1.100000,52.040000,2.967300e+04,-7.725460e+05,4.800000,0.000000,25.500000,1.010000,38.643333
25%,2009.750000,98.312500,5.500000,95.315000,2.796694e+06,-9.345000e+02,6.700000,39.225000,28.800000,1.390000,90.921667
50%,2014.500000,104.450000,8.500000,107.245000,7.350222e+06,9.459500e+03,7.200000,52.550000,30.100000,1.520000,100.000000
75%,2019.250000,114.385000,11.925000,132.242500,1.775614e+07,5.107825e+04,7.600000,61.200000,31.000000,1.710000,107.685000
max,2024.000000,263.140000,45.500000,1721.530000,5.198260e+08,1.751223e+06,8.100000,79.900000,33.200000,2.230000,879.781667


In [285]:
final_df['country'].nunique()

55

In [288]:
final_df["year"].value_counts().sort_index()

year
2005    55
2006    55
2007    55
2008    55
2009    55
2010    55
2011    55
2012    55
2013    55
2014    55
2015    55
2016    55
2017    55
2018    55
2019    55
2020    55
2021    55
2022    55
2023    55
2024    55
Name: count, dtype: int64

### Data Quality Assessment

## Data Preparation

### Data cleaning

### Missing values

### Duplicates

## Exploratory Data Analysis

### Univariate Analysis (Distributions)

 ### Trends Analysis

### Correlation Analysis

### Outlier Analysis

## Final Dataset Preparation

In [287]:
final_df.to_csv('../data/final_dataset_draft.csv', index=False)